## Setup

In [1]:
import os
import pandas as pd
from pathlib import Path
import spacy
from spacy.tokens import Doc
import skweak
import re
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)
from datetime import datetime
from datasets import Dataset
import numpy as np
import evaluate
import random
from blingfire import text_to_sentences
import pickle

In [2]:
from tqdm.notebook import tqdm

tqdm.pandas()

In [3]:
import IPython.display
import IPython.core.display

IPython.core.display.display = IPython.display.display  # type: ignore

In [4]:
# set up labeling stuff
label2id = {label: i for i, label in enumerate(["CONTRA", "NEUTRAL", "PRO"])}
id2label = {label2id[label]: label for label in label2id}

# model stuff
model_id = "FacebookAI/xlm-roberta-base"
seed = 42

## Utilities

In [5]:
def extract_speeches(
    root: str,
    years: list[str],
    *,
    filter_chairman: bool = True,
    filter_guest: bool = True,
    convert_dates: bool = False
) -> pd.DataFrame:
    """
    Read in the text data found in the ParliaMint dataset, from path to folder `root`,
    of the form `<PATH_TO_DATASET>\\ParlaMint-<LANG>.txt` and read in all speaches from
    years (represented as subfoders in the dataset) specified in `years`.
    """
    df_speech = pd.DataFrame()
    for year in years:
        path = Path(root) / year
        # loop through all the meta files
        for file in os.listdir(path):
            filename = os.fsdecode(file)
            if filename.endswith("meta-en.tsv"):
                # load using pandas
                df_meta = pd.read_csv(path / file, sep="\t", header=0)
                # load actual text
                df_text = pd.read_csv(
                    path / file.replace("-meta-en.tsv", ".txt"),
                    sep="\t",
                    names=["ID", "Text"],
                )
                # merge based on ID
                df_full = df_meta.merge(df_text, how="inner", on="ID")
                # concatenate with full dataframe
                df_speech = pd.concat([df_speech, df_full])

    # remove irrelevant entries
    # filter out stuff that I won't be processing
    if filter_chairman:
        df_speech = df_speech[(df_speech["Speaker_role"] != "Chairperson")]
    if filter_guest:
        df_speech = df_speech[(df_speech["Speaker_role"] != "Guest")]

    # do some data cleaning
    df_speech["Speaker_birth"] = pd.to_numeric(
        df_speech["Speaker_birth"], errors="coerce"
    )
    df_speech["Speaker_minister"] = df_speech["Speaker_minister"].map(
        lambda x: False if x == "notMinister" else True
    )
    df_speech["Speaker_MP"] = df_speech["Speaker_MP"].map(
        lambda x: True if x == "MP" else False
    )
    if convert_dates:
        df_speech["Date"] = pd.to_datetime(df_speech["Date"], format="%Y-%m-%d")

    # remove text annotations (clapping, shouting etc.)
    df_speech["Text"] = df_speech["Text"].apply(
        lambda x: re.sub(r"\[\[[^\[]*\]\]", "", x)
    )

    df_speech["Text"] = df_speech["Text"].progress_apply(
        lambda t: text_to_sentences(t).split("\n")
    )
    df_speech = df_speech.explode("Text").reset_index(drop=True)

    return df_speech

In [6]:
# [[ChatGPT]]
def nr_words(doc: Doc) -> int:
    """
    Count the number of words in the SpaCy Doc `doc`.
    """
    return len([token for token in doc if not token.is_punct and not token.is_space])

In [7]:
BATCH_SIZE = 128


def make_docs(nlp: spacy.language.Language, df: pd.DataFrame, text_column_name="Text"):
    """
    Make sentence-level documents from each text column entry of the dataframe `df`,
    using the `nlp` NLP model provided by spacy.
    """
    texts = df[text_column_name]
    metas = df.drop(columns=[text_column_name]).to_dict(orient="records")

    for doc, meta in tqdm(
        zip(nlp.pipe(texts, batch_size=BATCH_SIZE), metas),
        total=len(metas),
    ):

        doc._.attrs = meta
        yield doc

## Loading the Data

In [ ]:
df_speech = extract_speeches(".\\ParlaMint-HU.txt", ["2014", "2015", "2016"])
df_speech.head()

  0%|          | 0/18654 [00:00<?, ?it/s]

## Weak Supervision

### Preprocess Data

In [21]:
# load spacy stuff; disable all components to speed up processing
nlp = spacy.load(
    "hu_core_news_md",
    disable=[
        "tok2vec",
        "senter",
        "tagger",
        "morphologizer",
        "lookup_lemmatizer",
        "trainable_lemmatizer",
        "parser",
        "ner",
    ],
)
# set up spac Doc custom attribute
Doc.set_extension("attrs", default={}, force=True)


docs = list(make_docs(nlp, df_speech))
docs[:5]

  0%|          | 0/405096 [00:00<?, ?it/s]

[Tisztelt Országgyűlés!,
 Engedjék meg, hogy én is tisztelettel köszöntsem az alakuló ülés valamennyi résztvevőjét, vendégeinket és mindenkit, aki figyelemmel kíséri ezeket az ünnepélyes pillanatokat.,
 Bejelentem, hogy az országgyűlési törvény rendelkezései alapján a jegyzői feladatokat a tisztségviselők megválasztásáig a nyolc legfiatalabb képviselő: Bana Tibor, Csizi Péter, Demeter Márta, Dúró Dóra, Farkas Gergely, Heringes Anita, Kunhalmi Ágnes és Staudt Gábor képviselők korjegyzőként látják el. Ennek alapján felkérem Bana Tibor és Demeter Márta korjegyzőket, hogy legyenek most segítségemre az ülés vezetésében.,
 Tisztelt Köztársasági Elnök Úr!,
 Tisztelt Miniszterelnök Úr!]

### Define Labeling Functions

In [24]:
# helpers
def _lf_preprocess(doc: Doc):
    return doc.text.lower()

#### Pro-EA

In [25]:
# Strong leadership / centralisation framing [[ChatGPT]]
PRO_STRONG = [
    "erős kormány",
    "erős vezetés",
    "határozott kormányzás",
    "cselekvőképes kormány",
    "gyors döntéshozatal",
    "nemzeti érdek elsőbbsége",
]


def lf_pro_strong_leader(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in PRO_STRONG):
        yield doc[0].i, doc[-1].i + 1, "PRO"


lf_pro_strong_leader = skweak.heuristics.FunctionAnnotator(
    "lf_pro_strong_leader", lf_pro_strong_leader
)  # type: ignore

In [26]:
# Emergency powers justification [[ChatGPT]]
PRO_EMERGENCY = [
    "rendkívüli állapot szükséges",
    "veszélyhelyzet indokolja",
    "felhatalmazás szükséges",
    "különleges jogrend indokolt",
]


def lf_pro_emergency(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in PRO_EMERGENCY):
        yield doc[0].i, doc[-1].i + 1, "PRO"


lf_pro_emergency = skweak.heuristics.FunctionAnnotator(
    "lf_pro_emergency", lf_pro_emergency
)  # type: ignore

In [27]:
# Judiciary criticism [[ChatGPT]]
JUDICIARY = ["bíróság", "alkotmánybíróság", "bírók"]
NEG = ["akadályoz", "lassú", "politikai", "elfogult"]


def lf_pro_judiciary_attack(doc: Doc):
    text = _lf_preprocess(doc)
    if any(j in text for j in JUDICIARY) and any(n in text for n in NEG):
        yield doc[0].i, doc[-1].i + 1, "PRO"


lf_pro_judiciary_attack = skweak.heuristics.FunctionAnnotator(
    "lf_pro_judiciary_attack", lf_pro_judiciary_attack
)  # type: ignore

In [28]:
# Anti-opposition framing (weak proxy) [[ChatGPT]]
PRO_OPPOSITION_ATTACK = [
    "ellenzék akadályozza",
    "ellenzék hátráltatja",
    "ellenzék felelőtlen",
]


def lf_pro_opposition_attack(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in PRO_OPPOSITION_ATTACK):
        yield doc[0].i, doc[-1].i + 1, "PRO"


lf_pro_opposition_attack = skweak.heuristics.FunctionAnnotator(
    "lf_pro_opposition_attack", lf_pro_opposition_attack
)  # type: ignore

In [29]:
# Government party prior (from ParlaMint) [[ChatGPT]]
# TODO: Make this work


def lf_pro_government_party(doc):
    party = doc._.party  # from ParlaMint metadata
    gov_parties = doc._.gov_parties
    if party in gov_parties:
        return "PRO"

#### Contra-EA

In [30]:
# Rule of law / checks and balances [[ChatGPT]]
CON_RULE_OF_LAW = [
    "jogállamiság",
    "fékek és ellensúlyok",
    "hatalmi ágak szétválasztása",
    "alkotmányos kontroll",
]


def lf_con_rule_of_law(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in CON_RULE_OF_LAW):
        yield doc[0].i, doc[-1].i + 1, "CONTRA"


lf_con_rule_of_law = skweak.heuristics.FunctionAnnotator(
    "lf_con_rule_of_law", lf_con_rule_of_law
)  # type: ignore

In [31]:
# Democracy violation accusations[[ChatGPT]]
CON_DEMOCRACY_ATTACK = [
    "aláássa a demokráciát",
    "lebontja a demokráciát",
    "autoriter",
    "diktatórikus",
    "hatalomkoncentráció",
]


def lf_con_democracy_attack(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in CON_DEMOCRACY_ATTACK):
        yield doc[0].i, doc[-1].i + 1, "CONTRA"


lf_con_democracy_attack = skweak.heuristics.FunctionAnnotator(
    "lf_con_democracy_attack", lf_con_democracy_attack
)  # type: ignore

In [32]:
# Defense of judiciary / institutions [[ChatGPT]]
def lf_con_institution_defense(doc: Doc):
    text = _lf_preprocess(doc)
    if "védeni kell" in text or "meg kell védeni" in text:
        if "bíróság" in text or "alkotmány" in text:
            yield doc[0].i, doc[-1].i + 1, "CONTRA"


lf_con_institution_defense = skweak.heuristics.FunctionAnnotator(
    "lf_con_institution_defense", lf_con_institution_defense
)  # type: ignore

In [33]:
# Government overreach framing [[ChatGPT]]
CON_OVERREACH = ["kormány túlkapás", "hatalommal való visszaélés", "önkényes döntés"]


def lf_con_overreach(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in CON_OVERREACH):
        yield doc[0].i, doc[-1].i + 1, "CONTRA"


lf_con_overreach = skweak.heuristics.FunctionAnnotator(
    "lf_con_overreach", lf_con_overreach
)  # type: ignore

In [34]:
#  Opposition party prior [[ChatGPT]]
# TODO: Make this work
def lf_con_opposition_party(doc):
    party = doc._.party
    gov_parties = doc._.gov_parties
    if party not in gov_parties:
        return "CON"

#### Neutral

In [35]:
# Procedural speech [[ChatGPT]]
NEUTRAL_PROCEDURAL = [
    "napirendi pont",
    "köszönöm a szót",
    "szavazás következik",
    "jegyzőkönyv szerint",
    "ülést megnyitom",
]


def lf_neutral_procedural(doc: Doc):
    text = _lf_preprocess(doc)
    if any(p in text for p in NEUTRAL_PROCEDURAL):
        yield doc[0].i, doc[-1].i + 1, "NEUTRAL"


lf_neutral_procedural = skweak.heuristics.FunctionAnnotator(
    "lf_neutral_procedural", lf_neutral_procedural
)  # type: ignore

In [36]:
# Administrative / descriptive [[ChatGPT]]
def lf_neutral_descriptive(doc: Doc):
    text = _lf_preprocess(doc)
    if "jelentés szerint" in text or "statisztika alapján" in text:
        yield doc[0].i, doc[-1].i + 1, "NEUTRAL"


lf_neutral_descriptive = skweak.heuristics.FunctionAnnotator(
    "lf_neutral_descriptive", lf_neutral_descriptive
)  # type: ignore

In [37]:
# No political actors mentioned [[ChatGPT]]
# NOTE: I'm not convinced about this one
POLITICAL_TERMS = ["kormány", "miniszterelnök", "bíróság", "alkotmány", "ellenzék"]


def lf_neutral_no_politics(doc: Doc):
    text = _lf_preprocess(doc)
    if not any(p in text for p in POLITICAL_TERMS):
        yield doc[0].i, doc[-1].i + 1, "NEUTRAL"


lf_neutral_no_politics = skweak.heuristics.FunctionAnnotator(
    "lf_neutral_no_politics", lf_neutral_no_politics
)  # type: ignore

In [38]:
# Very short utterances [[ChatGPT]]
MIN_WORD_COUNT = 5


def lf_neutral_short(doc: Doc):
    if len(doc) < MIN_WORD_COUNT:
        yield doc[0].i, doc[-1].i + 1, "NEUTRAL"


lf_neutral_short = skweak.heuristics.FunctionAnnotator(
    "lf_neutral_short", lf_neutral_short
)  # type: ignore

In [39]:
# Quote / attribution detection [[ChatGPT]]
def lf_neutral_quote(doc: Doc):
    text = _lf_preprocess(doc)
    if "az ellenzék szerint" in text or "idézi" in text:
        yield doc[0].i, doc[-1].i + 1, "NEUTRAL"


lf_neutral_quote = skweak.heuristics.FunctionAnnotator(
    "lf_neutral_quote", lf_neutral_quote
)  # type: ignore

#### Combined

In [40]:
pro_annotator = skweak.base.CombinedAnnotator()
pro_annotator.add_annotators(
    lf_pro_emergency,
    lf_pro_opposition_attack,
    lf_pro_judiciary_attack,
    lf_pro_strong_leader,
)
contra_annotator = skweak.base.CombinedAnnotator()
contra_annotator.add_annotators(
    lf_con_democracy_attack,
    lf_con_institution_defense,
    lf_con_overreach,
    lf_con_rule_of_law,
)
neutral_annotator = skweak.base.CombinedAnnotator()
neutral_annotator.add_annotators(
    lf_neutral_descriptive,
    lf_neutral_no_politics,
    lf_neutral_procedural,
    lf_neutral_quote,
    lf_neutral_short,
)

In [41]:
full_annotator = skweak.base.CombinedAnnotator()
full_annotator.add_annotator(pro_annotator)
full_annotator.add_annotator(contra_annotator)
full_annotator.add_annotator(neutral_annotator)

### Run the Weak Labeling

In [42]:
# apply the annotator to my docs
annotated_docs = list(full_annotator.pipe(docs))

In [43]:
# consolidate it using a model
hmm = skweak.generative.HMM("hmm", labels=["PRO", "CONTRA", "NEUTRAL"])
hmm.fit(annotated_docs)

Starting iteration 1
Number of processed documents: 1000
Number of processed documents: 2000
Number of processed documents: 3000
Number of processed documents: 4000
Number of processed documents: 5000
Number of processed documents: 6000
Number of processed documents: 7000
Number of processed documents: 8000
Number of processed documents: 9000
Number of processed documents: 10000
Number of processed documents: 11000
Number of processed documents: 12000
Number of processed documents: 13000
Number of processed documents: 14000
Number of processed documents: 15000
Number of processed documents: 16000
Number of processed documents: 17000
Number of processed documents: 18000
Number of processed documents: 19000
Number of processed documents: 20000
Number of processed documents: 21000
Number of processed documents: 22000
Number of processed documents: 23000
Number of processed documents: 24000
Number of processed documents: 25000
Number of processed documents: 26000
Number of processed docume

         1 -2405017.21573276             +nan


Number of processed documents: 1000
Number of processed documents: 2000
Number of processed documents: 3000
Number of processed documents: 4000
Number of processed documents: 5000
Number of processed documents: 6000
Number of processed documents: 7000
Number of processed documents: 8000
Number of processed documents: 9000
Number of processed documents: 10000
Number of processed documents: 11000
Number of processed documents: 12000
Number of processed documents: 13000
Number of processed documents: 14000
Number of processed documents: 15000
Number of processed documents: 16000
Number of processed documents: 17000
Number of processed documents: 18000
Number of processed documents: 19000
Number of processed documents: 20000
Number of processed documents: 21000
Number of processed documents: 22000
Number of processed documents: 23000
Number of processed documents: 24000
Number of processed documents: 25000
Number of processed documents: 26000
Number of processed documents: 27000
Number of 

         2   -2027.24164278 +2402989.97408998


Number of processed documents: 1000
Number of processed documents: 2000
Number of processed documents: 3000
Number of processed documents: 4000
Number of processed documents: 5000
Number of processed documents: 6000
Number of processed documents: 7000
Number of processed documents: 8000
Number of processed documents: 9000
Number of processed documents: 10000
Number of processed documents: 11000
Number of processed documents: 12000
Number of processed documents: 13000
Number of processed documents: 14000
Number of processed documents: 15000
Number of processed documents: 16000
Number of processed documents: 17000
Number of processed documents: 18000
Number of processed documents: 19000
Number of processed documents: 20000
Number of processed documents: 21000
Number of processed documents: 22000
Number of processed documents: 23000
Number of processed documents: 24000
Number of processed documents: 25000
Number of processed documents: 26000
Number of processed documents: 27000
Number of 

         3   -2027.18744655      +0.05419623


Number of processed documents: 1000
Number of processed documents: 2000
Number of processed documents: 3000
Number of processed documents: 4000
Number of processed documents: 5000
Number of processed documents: 6000
Number of processed documents: 7000
Number of processed documents: 8000
Number of processed documents: 9000
Number of processed documents: 10000
Number of processed documents: 11000
Number of processed documents: 12000
Number of processed documents: 13000
Number of processed documents: 14000
Number of processed documents: 15000
Number of processed documents: 16000
Number of processed documents: 17000
Number of processed documents: 18000
Number of processed documents: 19000
Number of processed documents: 20000
Number of processed documents: 21000
Number of processed documents: 22000
Number of processed documents: 23000
Number of processed documents: 24000
Number of processed documents: 25000
Number of processed documents: 26000
Number of processed documents: 27000
Number of 

         4   -2027.18844809      -0.00100154
Model is not converging.  Current: -2027.188448086581 is not greater than -2027.1874465452474. Delta is -0.0010015413336077472


Finished E-step with 348998 documents


In [44]:
# fit the labels
fitted_docs = list(hmm.pipe(annotated_docs))

In [ ]:
# save the fitted data
with open("fitted_docs.pickle", "wb") as file:
    pickle.dump(fitted_docs, file, protocol=pickle.HIGHEST_PROTOCOL)

KeyboardInterrupt: 

## Load Data for Fine-Tuning

In [46]:
# or simply assigned the fitted docs to the proper variable, if within the same context
fitted_docs_r = fitted_docs

## Fine-Tuning the Classifier

### Dataset Setup

In [47]:
# extract the labels for spans
def extract_label(doc) -> str | None:
    labels = [x.label_ for x in skweak.utils.get_spans(doc, ["hmm"])]
    return labels[0] if len(labels) == 1 else None


def create_dataset(docs: list[Doc], pro_frac=0.3, con_frac=0.3, neut_frac=0.4):
    if pro_frac + con_frac + neut_frac > 1:
        raise ValueError("Error: fraction sum should be 1!")

    data = [{"text": doc.text, "label": extract_label(doc)} for doc in docs]
    contra_data = [d for d in data if d["label"] == "CONTRA"]
    neutral_data = [d for d in data if d["label"] == "NEUTRAL"]
    pro_data = [d for d in data if d["label"] == "PRO"]

    nr_con = len(contra_data)
    nr_neut = len(neutral_data)
    nr_pro = len(pro_data)

    random.seed(seed)

    print(nr_con, nr_neut, nr_pro)
    # if there are less pro and con than neutral, balance that way
    if nr_pro * pro_frac + nr_con * con_frac < nr_neut * neut_frac:
        nr_neut = int((nr_pro + nr_con) / (pro_frac + con_frac) * neut_frac)

        neutral_data = random.sample(neutral_data, nr_neut)
    # otherwise, balance it the other way around
    else:
        nr_pro = int(nr_neut / neut_frac * pro_frac)
        nr_con = int(nr_neut / neut_frac * con_frac)

        pro_data = random.sample(pro_data, nr_pro)
        contra_data = random.sample(contra_data, nr_con)

    print(nr_con, nr_neut, nr_pro)
    # full data
    full_data = [*pro_data, *contra_data, *neutral_data]

    # TODO: Add real validation (or remove it completely)
    return Dataset.from_list(full_data).train_test_split(test_size=0.1, seed=seed)


dataset = create_dataset(fitted_docs_r)
dataset

81 348778 139
81 146 139


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 329
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 37
    })
})

In [48]:
# save the dataset
with open("dataset.pickle", "wb") as file:
    pickle.dump(dataset, file, protocol=pickle.HIGHEST_PROTOCOL)

In [49]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)


# TODO: Refactor this to be in-line with the classifier one
def preprocess_function(speeches):
    tokenized = tokenizer(speeches["text"], truncation=True)
    tokenized["labels"] = [label2id[label] for label in speeches["label"]]
    return tokenized


tokenized_speeches = dataset.map(preprocess_function, batched=True)
tokenized_speeches = tokenized_speeches.select_columns(
    ["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

In [50]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Evaluation Setup

In [51]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels),
        "precision": precision.compute(
            predictions=predictions, references=labels, average=None
        )["precision"].tolist(),
        "recall": recall.compute(
            predictions=predictions, references=labels, average=None
        )["recall"].tolist(),
        "f1": f1_metric.compute(
            predictions=predictions, references=labels, average=None
        )["f1"].tolist(),
    }

### Training the Model

In [52]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=len(id2label), id2label=id2label, label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [53]:
training_args = TrainingArguments(
    output_dir="xlm-r-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_speeches["train"],
    eval_dataset=tokenized_speeches["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [54]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.075674,{'accuracy': 0.4864864864864865},"[0.5714285714285714, 0.4666666666666667, 0.0]","[0.5, 0.9333333333333333, 0.0]","[0.5333333333333333, 0.6222222222222222, 0.0]"
2,No log,0.945283,{'accuracy': 0.6756756756756757},"[0.0, 0.9230769230769231, 0.5416666666666666]","[0.0, 0.8, 0.9285714285714286]","[0.0, 0.8571428571428571, 0.6842105263157895]"
3,No log,0.836827,{'accuracy': 0.6756756756756757},"[0.0, 0.9230769230769231, 0.5416666666666666]","[0.0, 0.8, 0.9285714285714286]","[0.0, 0.8571428571428571, 0.6842105263157895]"


c:\Users\Kalami\anaconda3\envs\SKWeak\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Kalami\anaconda3\envs\SKWeak\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Kalami\anaconda3\envs\SKWeak\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=63, training_loss=0.9771774534195189, metrics={'train_runtime': 111.5453, 'train_samples_per_second': 8.848, 'train_steps_per_second': 0.565, 'total_flos': 96515346113928.0, 'train_loss': 0.9771774534195189, 'epoch': 3.0})

## Classification Pipeline

In [ ]:
# load the model
model_location = "./xlm-r-sentiment/checkpoint-42"
# TODO: Maybe change the type to "text-classification" or something more appropriate
classifier = pipeline(
    "text-classification", model=model_location, dtype="auto", batch_size=BATCH_SIZE
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# load the pandas df to be classified
df_speech = extract_speeches(".\\ParlaMint-HU.txt", ["2017"])
dataset = Dataset.from_pandas(df_speech)
df_speech.head()

  0%|          | 0/6077 [00:00<?, ?it/s]

,Text_ID,ID,Title,Date,Body,Term,Session,Meeting,Sitting,Agenda,...,Speaker_party,Speaker_party_name,Party_status,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth,Topic,Text
0,ParlaMint-HU_2017-02-20,u2017-02-20-1,"Hungarian parliamentary corpus ParlaMint-HU, l...",2017-02-20,Unicameralism,7. ciklus,-,73. ülés,1. ülésnap,-,...,Fidesz-frakció,Parliamentary group of the Fidesz – Hungarian ...,Coalition,Right to far-right,OrbanViktor,"Orbán, Viktor",M,1963.0,Macroeconomics,Tisztelt Elnök Úr!
1,ParlaMint-HU_2017-02-20,u2017-02-20-1,"Hungarian parliamentary corpus ParlaMint-HU, l...",2017-02-20,Unicameralism,7. ciklus,-,73. ülés,1. ülésnap,-,...,Fidesz-frakció,Parliamentary group of the Fidesz – Hungarian ...,Coalition,Right to far-right,OrbanViktor,"Orbán, Viktor",M,1963.0,Macroeconomics,Tisztelt Képviselőtársaim!
2,ParlaMint-HU_2017-02-20,u2017-02-20-1,"Hungarian parliamentary corpus ParlaMint-HU, l...",2017-02-20,Unicameralism,7. ciklus,-,73. ülés,1. ülésnap,-,...,Fidesz-frakció,Parliamentary group of the Fidesz – Hungarian ...,Coalition,Right to far-right,OrbanViktor,"Orbán, Viktor",M,1963.0,Macroeconomics,Tisztelt Ház!
3,ParlaMint-HU_2017-02-20,u2017-02-20-1,"Hungarian parliamentary corpus ParlaMint-HU, l...",2017-02-20,Unicameralism,7. ciklus,-,73. ülés,1. ülésnap,-,...,Fidesz-frakció,Parliamentary group of the Fidesz – Hungarian ...,Coalition,Right to far-right,OrbanViktor,"Orbán, Viktor",M,1963.0,Macroeconomics,"Kezdődő országgyűlési ülésszakunk elején, alko..."
4,ParlaMint-HU_2017-02-20,u2017-02-20-1,"Hungarian parliamentary corpus ParlaMint-HU, l...",2017-02-20,Unicameralism,7. ciklus,-,73. ülés,1. ülésnap,-,...,Fidesz-frakció,Parliamentary group of the Fidesz – Hungarian ...,Coalition,Right to far-right,OrbanViktor,"Orbán, Viktor",M,1963.0,Macroeconomics,Ma négy dologról kell beszélnem.


In [ ]:
texts = dataset["Text"]
labels, scores = [], []


for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i : i + BATCH_SIZE]
    preds = classifier(batch, batch_size=BATCH_SIZE, truncation=True)
    labels.extend([p["label"] for p in preds])
    scores.extend([p["score"] for p in preds])

# dataset = dataset.add_column("label", labels)
# dataset = dataset.add_column("score", scores)

  0%|          | 0/482 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [16]:
dataset = dataset.add_column("label", labels)
dataset = dataset.add_column("score", scores)

## Convert to Nicer Representation

In [17]:
# convert to pandas
df_speech = dataset.to_pandas()

# index by Speaker_ID and timestamp
df_speech = df_speech.set_index(["Speaker_ID", "Date", "ID"]).sort_index()

df_speech.head()

Text_ID  \
Speaker_ID Date       ID                                         
-          2017-03-21 u2017-03-21-21   ParlaMint-HU_2017-03-21   
                      u2017-03-21-21   ParlaMint-HU_2017-03-21   
AghPeter   2017-03-13 u2017-03-13-137  ParlaMint-HU_2017-03-13   
                      u2017-03-13-137  ParlaMint-HU_2017-03-13   
                      u2017-03-13-137  ParlaMint-HU_2017-03-13   

                                                                                   Title  \
Speaker_ID Date       ID                                                                   
-          2017-03-21 u2017-03-21-21   Hungarian parliamentary corpus ParlaMint-HU, l...   
                      u2017-03-21-21   Hungarian parliamentary corpus ParlaMint-HU, l...   
AghPeter   2017-03-13 u2017-03-13-137  Hungarian parliamentary corpus ParlaMint-HU, l...   
                      u2017-03-13-137  Hungarian parliamentary corpus ParlaMint-HU, l...   
                      u2017-03-13-137  Hungarian parliamentary corpus ParlaMint-HU, l...   

                                                Body       Term Session  \
Speaker_ID Date       ID                                                  
-          2017-03-21 u2017-03-21-21   Unicameralism  7. ciklus       -   
                      u2017-03-21-21   Unicameralism  7. ciklus       -   
AghPeter   2017-03-13 u2017-03-13-137  Unicameralism  7. ciklus       -   
                      u2017-03-13-137  Unicameralism  7. ciklus       -   
                      u2017-03-13-137  Unicameralism  7. ciklus       -   

                                        Meeting     Sitting Agenda  Subcorpus  \
Speaker_ID Date       ID                                                        
-          2017-03-21 u2017-03-21-21   75. ülés  2. ülésnap      -  Reference   
                      u2017-03-21-21   75. ülés  2. ülésnap      -  Reference   
AghPeter   2017-03-13 u2017-03-13-137  74. ülés  4. ülésnap      -  Reference   
                      u2017-03-13-137  74. ülés  4. ülésnap      -  Reference   
                      u2017-03-13-137  74. ülés  4. ülésnap      -  Reference   

                                            Lang  ...  \
Speaker_ID Date       ID                          ...   
-          2017-03-21 u2017-03-21-21   Hungarian  ...   
                      u2017-03-21-21   Hungarian  ...   
AghPeter   2017-03-13 u2017-03-13-137  Hungarian  ...   
                      u2017-03-13-137  Hungarian  ...   
                      u2017-03-13-137  Hungarian  ...   

                                                                      Speaker_party_name  \
Speaker_ID Date       ID                                                                   
-          2017-03-21 u2017-03-21-21                                                   -   
                      u2017-03-21-21                                                   -   
AghPeter   2017-03-13 u2017-03-13-137  Parliamentary group of the Fidesz – Hungarian ...   
                      u2017-03-13-137  Parliamentary group of the Fidesz – Hungarian ...   
                      u2017-03-13-137  Parliamentary group of the Fidesz – Hungarian ...   

                                       Party_status   Party_orientation  \
Speaker_ID Date       ID                                                  
-          2017-03-21 u2017-03-21-21              -                   -   
                      u2017-03-21-21              -                   -   
AghPeter   2017-03-13 u2017-03-13-137     Coalition  Right to far-right   
                      u2017-03-13-137     Coalition  Right to far-right   
                      u2017-03-13-137     Coalition  Right to far-right   

                                      Speaker_name Speaker_gender  \
Speaker_ID Date       ID                                            
-          2017-03-21 u2017-03-21-21             -              -   
                      u2017-03-21-21             -              -   
AghPet

In [20]:
# save the data
df_speech.to_csv("./outputs/hu_2017.csv")

## Visualizations